# Eval

Needs `classification_train.csv` and `lora_adapter/` from 01–02. Macro-F1 vs a small GPT-4o-mini sample.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from peft import PeftModel
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer

_p = Path.cwd().resolve()
ROOT = next((a for a in [_p, *_p.parents] if (a / "pyproject.toml").is_file()), _p)
load_dotenv(ROOT / ".env")

csv_path = ROOT / "notebooks" / "finetune_classifier" / "classification_train.csv"
adapter = ROOT / "notebooks" / "finetune_classifier" / "lora_adapter"
df = pd.read_csv(csv_path)
labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)
_, test_df = train_test_split(df, test_size=0.15, stratify=df["label_id"], random_state=42)

base_name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(adapter)
base = AutoModelForSequenceClassification.from_pretrained(
    base_name, num_labels=len(labels), id2label=id2label, label2id=label2id
)
model = PeftModel.from_pretrained(base, str(adapter))
model.eval()
dev = next(model.parameters()).device

def predict_chunk(texts: list[str]) -> np.ndarray:
    enc = tok(texts, truncation=True, padding=True, max_length=256, return_tensors="pt").to(dev)
    with torch.no_grad():
        logits = model(**enc).logits
    return logits.argmax(-1).cpu().numpy()

chunks = [test_df["text"].iloc[i : i + 16].tolist() for i in range(0, len(test_df), 16)]
y_pred = np.concatenate([predict_chunk(c) for c in chunks])
y_true = test_df["label_id"].values
f1_ft = f1_score(y_true, y_pred, average="macro")
print("LoRA macro-F1:", f1_ft)
print(classification_report(y_true, y_pred, target_names=labels))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

client = OpenAI()
MODEL = os.environ.get("MEDFLOW_SYNTH_MODEL", "gpt-4o-mini")
gpt_labels: list[int] = []
sample = test_df.head(40)
for _, row in sample.iterrows():
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": "Reply with exactly one label from: " + ", ".join(labels)},
                  {"role": "user", "content": row["text"][:6000]}],
        temperature=0,
        max_tokens=20,
    )
    raw = (r.choices[0].message.content or "").strip()
    lab = raw.split()[0] if raw else "OTHER"
    gpt_labels.append(label2id.get(lab, 0))
y_s = sample["label_id"].values
f1_gpt = f1_score(y_s, np.array(gpt_labels), average="macro")
print("GPT-4o-mini macro-F1 (n=40):", f1_gpt)
summary = {"lora_macro_f1": float(f1_ft), "gpt4o_mini_macro_f1_sample40": float(f1_gpt)}
(ROOT / "notebooks" / "finetune_classifier" / "eval_summary.json").write_text(json.dumps(summary, indent=2))
print("Wrote eval_summary.json", summary)